In [ ]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point
import polars as pl

In [ ]:
import polars as pl
from pathlib import Path

logger = logging.getLogger()
logger.setLevel(logging.INFO)

if logger.hasHandlers():
    logger.handlers.clear()
file_handler = logging.FileHandler("calculating_features_2.log", mode="w")
formatter = logging.Formatter("%(asctime)s - %(levelname)s - %(message)s")
file_handler.setFormatter(formatter)
logger.addHandler(file_handler)


input_path=Path('benchmark/test_skfnac/vector/data_for_pict/time_divided_meteodata')
aggregated_path = Path('benchmark/test_skfnac/vector/data_for_pict/preds/aggregated_meteodata')

aggregated_path.mkdir(parents=True, exist_ok=True)

for file in input_path.glob('*.gzip'):
    cls=file.stem.rsplit('_')[5]
    out_file=aggregated_path.joinpath(file.name)
    if out_file.exists():
        logger.info(f'file {out_file.name} is already exists. Skipping...')
        continue
        
    df = pl.read_parquet(file)
    condition = df["temperature"] > 10
    
    com_calculated = (
        df.group_by(["lat", "lon", "month"])
        .agg([
            pl.col("temperature").median().alias("median_t"),
            pl.col("precipitation").median().alias("median_prec"),
            pl.col("precipitation").sum().alias("sum_prec")
        ])
    )
    
    temp_calculated = (
        df.filter(condition)
        .group_by(["lat", "lon", "month"])
        .agg([pl.col("temperature").sum().alias("sum_t")])
    )

    calculated_df = com_calculated.join(temp_calculated, on=["lat", "lon", "month"], how='left')
    
    result_df = None
    
    for col in ["median_t", "sum_t", "median_prec", "sum_prec"]:
        pivot_df = calculated_df.pivot(
            index=["lat", "lon"],
            on="month",
            values=col,
            aggregate_function=None
        ).rename({
            str(month): f"{col}_{month}" for month in calculated_df["month"].unique()
        })
        
        if result_df is None:
            result_df = pivot_df
        else:
            result_df = result_df.join(pivot_df, on=["lat", "lon"])
            result_df = result_df.with_columns(pl.lit(cls).alias('class'))
    result_df.write_parquet(out_file, compression='gzip')

In [ ]:
def hampel(y, window_size, simg=3):    
    n = len(y)
    new_y = y.copy()
    k = 1.4826
    idx = []

    for i in range((window_size),(n - window_size)):
        r_median = np.median(y[(i - window_size):(i + window_size)]) #скользящая медиана 
        r_mad  = np.median(np.abs(y[(i - window_size):(i + window_size)] - r_median)) #скользящий MAD 
        if (np.abs(y[i] - r_median) > simg * r_mad):
            new_y[i] = r_median #замена выброса
            # idx.append(i)
    
    return new_y

def process_group_means(group_df, x_values_for_derivative, index):
    """Process a single group of data (lat/lon combination)."""
    try:
        group_df = group_df.filter(pl.col("DOY") >= 90)
            
        months = group_df["DOY"].to_numpy().astype(np.float64)
        values = group_df[index].to_numpy().astype(np.float64)
        
        # processed_values = lowess_transform(months, values, 0.3)
        processed_values=hampel(values, 3)
        
        f = interp1d(months, processed_values, kind='linear', fill_value="extrapolate")
        interpolated_values = f(x_values_for_derivative)
        
        doy_max = x_values_for_derivative[np.argmax(interpolated_values)]
        doy_min = x_values_for_derivative[np.argmin(interpolated_values)]
                        
        result_dict = {
            'field_id': group_df['field_id'][0],
            f'{index}_min': np.min(interpolated_values),
            f'{index}_doy_min': doy_min,
            f'{index}_doy_max': doy_max,
            f'{index}_max': np.max(interpolated_values),
        }

        # Calculate monthly statistics
        synth_df = pl.DataFrame({
            'DOY': x_values_for_derivative,
            f'{index}': interpolated_values
        })
        synth_df = synth_df.with_columns(
            (pl.col("DOY").cast(pl.Date).dt.month().alias("month"))
        )
        
        monthly_stats = synth_df.group_by("month").agg([
            pl.col(f"{index}").median().alias(f"{index}_median_fitted")
        ])
        
        for month in range(4, 11):
            month_data = monthly_stats.filter(pl.col("month") == month)
            if len(month_data) > 0:
                result_dict[f'median_{index}_fitted_{month}'] = month_data[f'{index}_median_fitted'][0]
            else:
                result_dict[f'median_{index}_fitted_{month}'] = None
                
        return result_dict
        
    except Exception as e:
        logger.error(f"Error processing group: {e}")
        return None

def process_file_means(file_path, output_path, index, x_values_for_derivative):
    """Process a single input file."""
    try:
        output_file = output_path / index / file_path.name
        if output_file.exists():
            logger.info(f'File {file_path.name} already exists. Skipping...')
            return

        out_folder=output_path / index
        results=[]

        if not out_folder.exists():
            out_folder.mkdir(parents=True, exist_ok=True)
            
        logger.info(f'Processing {file_path.name}...')
        df = pl.read_parquet(file_path)
        
        for field, group in df.group_by(["field_id"]):
            result=process_group_means(group, x_values_for_derivative, index)
            results.append(result)
        
    
        if results:
            result_df = pl.DataFrame(results)
            result_df.write_parquet(output_file, compression="gzip")
            logger.info(f'File {file_path.name} successfully saved...')
        else:
            logger.info(f'No valid results for {file_path.name}')
            
    except Exception as e:
        logger.error(f"Error processing file {file_path.name}: {e}")


logger = logging.getLogger()
logger.setLevel(logging.INFO)
logger.setLevel(logging.DEBUG)

if logger.hasHandlers():
    logger.handlers.clear()
file_handler = logging.FileHandler("calculating_features_2.log", mode="w")
formatter = logging.Formatter("%(asctime)s - %(levelname)s - %(message)s")
file_handler.setFormatter(formatter)
logger.addHandler(file_handler)

x_values_for_derivative = np.linspace(1, 365, 2000)
time_path = Path('benchmark/test_skfnac/vector/data_for_pict/time_divided')
output_path = Path('benchmark/test_skfnac/vector/data_for_pict/preds')

output_path.mkdir(parents=True, exist_ok=True)
indices = ['red', 'nir', 'blue', 'swir1', 'green', 'swir2', 
           'ndyi']
for file in time_path.glob('*.gzip'):
    for index in indices:
        process_file_means(file, output_path, index, x_values_for_derivative)

In [ ]:
def hampel(y, window_size, simg=3):    
    n = len(y)
    new_y = y.copy()
    k = 1.4826
    idx = []

    for i in range((window_size),(n - window_size)):
        r_median = np.median(y[(i - window_size):(i + window_size)]) #скользящая медиана 
        r_mad  = np.median(np.abs(y[(i - window_size):(i + window_size)] - r_median)) #скользящий MAD 
        if (np.abs(y[i] - r_median) > simg * r_mad):
            new_y[i] = r_median
    return new_y

def double_logistic_function(t, wNDVI, mNDVI, S, A, mS, mA):
    """Double logistic function for curve fitting."""
    sigmoid1 = 1 / (1 + np.exp(-mS * (t - S)))
    sigmoid2 = 1 / (1 + np.exp(mA * (t - A)))
    seasonal_term = sigmoid1 + sigmoid2 - 1
    return wNDVI + (mNDVI - wNDVI) * seasonal_term

def weight_function(t, S, A, r):
    """Weight function for curve fitting."""
    tr = 100 * (t - S) / (A - S)
    tr = np.clip(tr, 0, 100)
    return np.exp(-np.abs(r) / (1 + tr / 10))

def fit_curve(t, ndvi_observed, bounds):
    """Fit the double logistic curve to the data."""
    initial_guess = [
        np.min(ndvi_observed),
        np.max(ndvi_observed),
        0,
        365,
        0.1,
        0.1
    ]
    
    try:
        params, _ = curve_fit(
            double_logistic_function, t, ndvi_observed,
            p0=initial_guess, bounds=bounds, maxfev=5000
        )
        
        residuals = ndvi_observed - double_logistic_function(t, *params)
        weights = weight_function(t, params[2], params[3], residuals)
        
        params, _ = curve_fit(
            double_logistic_function, t, ndvi_observed,
            p0=params, bounds=bounds, sigma=weights, maxfev=5000
        )
        return params
    except Exception as e:
        logger.error(f"Curve fitting failed: {e}")
        return None

def calculate_quality_metrics(observed, fitted):
    """Calculate R-squared, MSE, and RMSE."""
    residuals = observed - fitted
    ss_res = np.sum(residuals**2)
    ss_tot = np.sum((observed - np.mean(observed))**2)
    r_squared = 1 - (ss_res / ss_tot) if ss_tot != 0 else 0
    mse = np.mean(residuals**2)
    rmse = np.sqrt(mse)
    return r_squared, mse, rmse

def sort_extrema_points(extrema_points_x, doy_max):
    """Sort extremum points based on comparison with doy_max."""
    less_than_doy_max = [x for x in extrema_points_x if x < doy_max]
    greater_than_doy_max = [x for x in extrema_points_x if x > doy_max]
    
    return {
        'start_of_growth': min(less_than_doy_max) if less_than_doy_max else None,
        'end_of_growth': max(less_than_doy_max) if less_than_doy_max else None,
        'start_of_decay': min(greater_than_doy_max) if greater_than_doy_max else None,
        'end_of_decay': max(greater_than_doy_max) if greater_than_doy_max else None
    }

def process_group(group_df, x_values_for_derivative, symbols_for_lambdify, fourth_derivative_sym, index, bounds):
    """Process a single group of data (lat/lon combination)."""
    try:
        group_df = group_df.filter(pl.col("DOY") >= 90)
        if len(group_df) < 6:
            return None
            
        months = group_df["DOY"].to_numpy().astype(np.float64)
        values = group_df[index].to_numpy().astype(np.float64)

        
        # processed_values = lowess_transform(months, values, 0.3)
        processed_values=hampel(values, 3)
        
        params = fit_curve(months, processed_values, bounds)
        if params is None or np.isnan(params).any():
            return None
        
        wNDVI, mNDVI, S, A, mS, mA = params
    
        if (mNDVI < wNDVI) or (A < S) or (int(A-S)<20) or (A == S):
            return None
            
        fitted_values = double_logistic_function(months, *params)
        r_squared, mse, rmse = calculate_quality_metrics(processed_values, fitted_values)
        
        finer_values = double_logistic_function(x_values_for_derivative, *params)
        doy_max = x_values_for_derivative[np.argmax(finer_values)]
                
        f_quadruple_prime_lambdified = sp.lambdify(symbols_for_lambdify, fourth_derivative_sym, 'numpy')
        fourth_derivative_values = f_quadruple_prime_lambdified(x_values_for_derivative, *params)
        zero_crossings_fourth = np.where(np.diff(np.sign(fourth_derivative_values)))[0]
        extrema_points_third_x = x_values_for_derivative[zero_crossings_fourth]
        extrema_points_third_x = sorted(set(extrema_points_third_x))
        
        sorted_points = sort_extrema_points(extrema_points_third_x, doy_max)
        
        result_dict = {
            'field_id': group_df['field_id'][0],
            f'{index}_wNDVI': wNDVI,
            f'{index}_mNDVI': mNDVI,
            f'{index}_S': S,
            f'{index}_A': A,
            f'{index}_mS': mS,
            f'{index}_mA': mA,
            f'{index}_doy_max': doy_max,
            f'{index}_max': double_logistic_function(doy_max, *params),
            f'{index}_start_of_growth': sorted_points.get('start_of_growth'),
            f'{index}_end_of_growth': sorted_points.get('end_of_growth'),
            f'{index}_start_of_decay': sorted_points.get('start_of_decay'),
            f'{index}_end_of_decay': sorted_points.get('end_of_decay'),
            f'{index}_max_growth': double_logistic_function(sorted_points['end_of_growth'], *params) if sorted_points['end_of_growth'] else None,
            f'{index}_mean_growth': double_logistic_function(S, *params),
            f'{index}_min_growth': double_logistic_function(sorted_points['start_of_growth'], *params) if sorted_points['start_of_growth'] else None,
            f'{index}_max_decay': double_logistic_function(sorted_points['start_of_decay'], *params) if sorted_points['start_of_decay'] else None,
            f'{index}_min_decay': double_logistic_function(sorted_points['end_of_decay'], *params) if sorted_points['end_of_decay'] else None,
            f'{index}_mean_decay': double_logistic_function(A, *params),
        }

        
        # Calculate monthly statistics
        synth_df = pl.DataFrame({
            'DOY': x_values_for_derivative,
            f'{index}': finer_values
        })
        synth_df = synth_df.with_columns(
            (pl.col("DOY").cast(pl.Date).dt.month().alias("month"))
        )
        
        monthly_stats = synth_df.group_by("month").agg([
            pl.col(f"{index}").median().alias(f"{index}_median_fitted")
        ])
        
        for month in range(4, 11):
            month_data = monthly_stats.filter(pl.col("month") == month)
            if len(month_data) > 0:
                result_dict[f'median_{index}_fitted_{month}'] = month_data[f'{index}_median_fitted'][0]
            else:
                result_dict[f'median_{index}_fitted_{month}'] = None
                
        return result_dict
        
    except Exception as e:
        logger.error(f"Error processing group: {e}")
        return None

def process_file(file_path, output_path, x_values_for_derivative, symbols_for_lambdify, fourth_derivative_sym, index, bounds):
    """Process a single input file."""
    try:
        output_file = output_path / index / file_path.name
        if output_file.exists():
            logger.info(f'File {file_path.name} already exists. Skipping...')
            return

        out_folder=output_path / index

        if not out_folder.exists():
            out_folder.mkdir(parents=True, exist_ok=True)
            
        logger.info(f'Processing {file_path.name}...')
        df = pl.read_parquet(file_path)
        
        groups = df.group_by(["field_id"])
        tasks = [delayed(process_group)(group_df, x_values_for_derivative, symbols_for_lambdify, fourth_derivative_sym, index, bounds) 
                 for group_key, group_df in groups]
        results_list = Parallel(n_jobs=multiprocessing.cpu_count(), verbose=10, backend='loky')(tasks) 
        
        results = [res for res in results_list if res is not None]
        

        if results:
            result_df = pl.DataFrame(results)
            result_df.write_parquet(output_file, compression="gzip")
            logger.info(f'File {file_path.name} successfully saved...')
        else:
            logger.info(f'No valid results for {file_path.name}')
            
    except Exception as e:
        logger.error(f"Error processing file {file_path.name}: {e}")


logger = logging.getLogger()
logger.setLevel(logging.INFO)
logger.setLevel(logging.DEBUG)

if logger.hasHandlers():
    logger.handlers.clear()
file_handler = logging.FileHandler("calculating_features_2.log", mode="w")
formatter = logging.Formatter("%(asctime)s - %(levelname)s - %(message)s")
file_handler.setFormatter(formatter)
logger.addHandler(file_handler)

time_path = Path('benchmark/test_skfnac/vector/data_for_pict/time_divided')
output_path = Path('benchmark/test_skfnac/vector/data_for_pict/preds')
output_path.mkdir(parents=True, exist_ok=True)

x = sp.symbols('x')
wNDVI_sym, mNDVI_sym, S_sym, A_sym, mS_sym, mA_sym = sp.symbols('wNDVI mNDVI S A mS mA')
sigmoid1_sym = 1 / (1 + sp.exp(-mS_sym * (x - S_sym)))
sigmoid2_sym = 1 / (1 + sp.exp(mA_sym * (x - A_sym)))
seasonal_term_sym = sigmoid1_sym + sigmoid2_sym - 1
sympy_dlf_template = wNDVI_sym + (mNDVI_sym - wNDVI_sym) * seasonal_term_sym
first_derivative_sym = sp.diff(sympy_dlf_template, x)
second_derivative_sym = sp.diff(first_derivative_sym, x)
third_derivative_sym = sp.diff(second_derivative_sym, x)
fourth_derivative_sym = sp.diff(third_derivative_sym, x)

symbols_for_lambdify = [x, wNDVI_sym, mNDVI_sym, S_sym, A_sym, mS_sym, mA_sym]
x_values_for_derivative = np.linspace(1, 365, 2000)


from itertools import product

indices = ['wrdvi', 'ndre', 'sipi', 'ndvi']
bounds_config = {
    'wrdvi': {
        1: ([-1, -1, -180, -180, 0, 0], [1, 1, 365, 365, 1, 1]),
        'default': ([-1, -1, 0, 0, 0, 0], [1, 1, 365, 365, 1, 1])
    },
    'ndvi': {
        1: ([-1, -1, -180, -180, 0, 0], [1, 1, 365, 365, 1, 1]),
        'default': ([-1, -1, 0, 0, 0, 0], [1, 1, 365, 365, 1, 1])
    },
    'ndre': {
        1: ([-0.2, -0.2, -180, -180, 0, 0], [1, 1, 365, 365, 1, 1]),
        'default': ([-0.2, -0.2, 0, 0, 0, 0], [1, 1, 365, 365, 1, 1])
    },
    'sipi': {
        1: ([0, 0, -180, -180, 0, 0], [1, 1, 365, 365, 1, 1]),
        'default': ([0, 0, 0, 0, 0, 0], [1, 1, 365, 365, 1, 1])
    }
}
cl=2
for file in time_path.glob('*.gzip'):
    for index in indices:
        bounds = bounds_config[index].get(cl, bounds_config[index]['default'])
        process_file(file, output_path, x_values_for_derivative, symbols_for_lambdify, fourth_derivative_sym, index, bounds)

In [ ]:
meteopath=Path('benchmark/test_skfnac/vector/data_for_pict/preds/aggregated_meteodata')
fields_path=Path('benchmark/test_skfnac/vector/shp_for_apply')
output_path=Path('benchmark/test_skfnac/vector/data_for_pict/preds/aggregated_meteodata_with_fields')

for file in meteopath.glob('*.gzip'):
    year=file.stem.rsplit('_')[3]
    cls=file.stem.rsplit('_')[5]
    df=pd.read_parquet(file)
    gdf_points = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df.lon, df.lat),
        crs='EPSG:4326'
    )
    
    gdf_polygons = gpd.read_file(fields_path.joinpath(f'{year}.shp'))
    
    if gdf_points.crs != gdf_polygons.crs:
        gdf_points = gdf_points.to_crs(gdf_polygons.crs)
    
    gdf_joined = gpd.sjoin(gdf_points, gdf_polygons[['field_id', 'geometry']], how="left", predicate='within')
    
    df['field_id'] = gdf_joined['field_id'].values
    df.to_parquet(output_path.joinpath(f'{cls}_{year}.parquet.gzip'), index=False)

In [ ]:
folders = ['wrdvi', 'ndre', 'sipi', 'ndyi', 'red', 'nir', 'blue', 'ndvi', 'swir1', 'green', 'swir2', 'aggregated_meteodata_with_fields']

meteo_path=Path('benchmark/test_skfnac/vector/data_for_pict/preds/aggregated_meteodata_with_fields')
base_path = Path('benchmark/test_skfnac/vector/data_for_pict/preds')
output_path=Path('benchmark/test_skfnac/vector/data_for_pict/preds/aggregated_data')
output_path.mkdir(parents=True, exist_ok=True)

files = base_path.glob(f'wrdvi/*.gzip')

for file_name in files:
    dfs = []
    for folder in folders:
        file_path = base_path.joinpath(f'{folder}/{file_name.name}')
        if not file_path.exists():
            continue

        df = pl.read_parquet(file_path)
        if 'R_squared' in df.columns:
            df=df.drop(['R_squared', 'MSE', 'RMSE'])
        
        cols_to_keep = [c for c in df.columns if c not in ['field_id']]
        df = df.select(['field_id'] + cols_to_keep)
        dfs.append(df)
    
    merged_df = dfs[0]
    for df in dfs[1:]:
        merged_df = merged_df.join(df, on=['field_id'], how='inner')
        
    meteo_list=[pl.read_parquet(meteo_file) for meteo_file in meteo_path.glob(f'*{year}.parquet.gzip')]
    all_columns = set()
    for df in meteo_list:
        all_columns.update(df.columns)
    all_columns = list(set.union(*(set(df.columns) for df in meteo_list)))
    first_types = {col: meteo_list[0].schema.get(col, pl.Float64) for col in all_columns}
    
    fixed_meteo_list = []
    for df in meteo_list:
        for col in all_columns:
            if col not in df.columns:
                df = df.with_columns(pl.lit(None).cast(first_types[col]).alias(col))
            else:
                df = df.with_columns(df[col].cast(first_types[col]))
        df = df.select(all_columns)
        fixed_meteo_list.append(df)
    
    concatenated_meteo = pl.concat(fixed_meteo_list, how='vertical')
    
    concatenated_meteo=concatenated_meteo.with_columns(pl.col('field_id').cast(pl.Int64, strict=False))
    merged_df = merged_df.join(concatenated_meteo, on=['field_id'], how='inner')

        
    if not merged_df.is_empty():
        merged_df.write_parquet(output_path.joinpath(file_name.name))